# G0 — Do the global off and unity runs give the same results?

Complete for the two-day comparison; last update: 8 October 2026.

The full G0 checker found exact equality in all three histories and both restarts. This notebook makes that result easier to read by showing Arctic and Antarctic totals and statistics from the final daily mean. It does not replace the full comparison.

No conceptual figure or plotting-library import is needed. NetCDF processing runs in a separate process and reads small row chunks, so a native-library failure is reported by that process rather than taking down the notebook's kernel. All figures added later should use PyGMT.


In [1]:
from pathlib import Path
import os, sys, json, csv, html, subprocess
from IPython.display import display, HTML

repo = Path(os.environ.get("CICE_MODEL_REPO", "/g/data/gv90/da1339/src/CICE_dyntens")).expanduser().resolve()
runs = Path(os.environ.get("CICE_TEST_RUNS", "/g/data/gv90/da1339/cice-dirs/runs")).expanduser().resolve()
output = repo / "validation_report/global/g0_summary"

# Final daily mean; the instantaneous stop is represented by the final restart.
history_name = "iceh.2000-09-02.nc"
off = runs / "dt_g0_off/history" / history_name
unity = runs / "dt_g0_unity/history" / history_name
latitude_cutoff = 0.0  # NH > 0 degrees, SH < 0 degrees; change only with a stated regional definition.
ice_threshold = 0.15 # Extent and selected-cell field statistics.
rows_per_chunk = 32

if not (repo / "cice.setup").is_file():
    raise FileNotFoundError("Set CICE_MODEL_REPO to your CICE_dyntens checkout")
for path in (off, unity):
    if not path.is_file():
        raise FileNotFoundError(path)
print("Off:", off)
print("Unity:", unity)


Off: /g/data/gv90/da1339/cice-dirs/runs/dt_g0_off/history/iceh.2000-09-02.nc
Unity: /g/data/gv90/da1339/cice-dirs/runs/dt_g0_unity/history/iceh.2000-09-02.nc


## Full comparison record

Use the G0 workflow to check the entire output first. This notebook reads its result; it does not award a new PASS from matching regional averages. The supplied G0 comparison passed, but the local evidence file must also show PASS before these tables are generated.


In [2]:
evidence = repo / "validation_report/global/evidence/g0-short-validation.json"
if not evidence.is_file():
    raise FileNotFoundError("Run g0_global_workflow.py analyse --evidence validation_report/global/evidence first")
record = json.loads(evidence.read_text())
if record.get("status") != "PASS" or record.get("gate") != "G0-short":
    raise ValueError("The local full G0 comparison is not recorded as PASS")
print("Full G0 comparison:", record["status"])


Full G0 comparison: PASS


## Read the final daily mean

This cell may take a little time to read and hash both files. It uses the notebook's Python environment in a separate process; that environment needs NumPy and netCDF4, but not GMT/PyGMT. If it fails, the error and process return code are shown here. The kernel can remain running.


In [3]:
command = [sys.executable, "-m", "CICE_testing.validation.global_summary",
           "--off", str(off), "--unity", str(unity), "--output", str(output),
           "--latitude-cutoff", str(latitude_cutoff), "--ice-threshold", str(ice_threshold),
           "--rows", str(rows_per_chunk)]
result = subprocess.run(command, cwd=repo, capture_output=True, text=True)
print(result.stdout)
if result.returncode:
    print(result.stderr)
    raise RuntimeError(f"Summary process exited with code {result.returncode}; see its error above")
summary = json.loads((output / "g0-regional-summary.json").read_text())
print("Time and interval:", summary["time"])
print("Fields not summarised:", "; ".join(summary["skipped_fields"]))


WROTE /g/data/gv90/da1339/src/CICE_dyntens/validation_report/global/g0_summary/g0-regional-totals.csv
WROTE /g/data/gv90/da1339/src/CICE_dyntens/validation_report/global/g0_summary/g0-regional-fields.csv

Time and interval: {'units': 'days since 2000-09-01 00:00:00', 'calendar': 'proleptic_gregorian', 'value': 2.0, 'date': '2000-09-03 00:00:00', 'bounds': [1.0, 2.0], 'interval': ['2000-09-02 00:00:00', '2000-09-03 00:00:00']}
Fields not summarised: hs: not written; Tsfc: not written


## Circum-Arctic and circum-Antarctic totals

Here these names mean all ocean cells in the northern and southern hemispheres, respectively. Land and the equator are excluded. Using all hemispheric ocean cells includes the surrounding seasonal ice seas; a different latitude cutoff defines a different comparison.

Ice area = sum(cell area × ice concentration). Ice extent = sum(cell area where daily mean concentration is at least 15%). Ice volume = sum(cell area × `hi`), where `hi` is ice volume per grid-cell area. Mean ice-covered thickness = total volume / total ice area.

These quantities use the final **daily mean**, not an instantaneous stop-time state. The table should show zero unity-minus-off differences.


In [4]:
def show_csv(path):
    with path.open(newline="") as stream:
        reader = csv.DictReader(stream)
        rows = list(reader)
        columns = reader.fieldnames or []
    def formatted(value):
        if value in (None, ""):
            return "—"
        try:
            return f"{float(value):.8g}"
        except ValueError:
            return value
    head = "".join(f"<th>{html.escape(c)}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{html.escape(formatted(row[c]))}</td>" for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))
    print("CSV:", path)
    return rows

totals = show_csv(output / "g0-regional-totals.csv")


region,metric,units,off,unity,unity_minus_off
Circum-Arctic (NH),area,km2,4628638.4,4628638.4,0
Circum-Arctic (NH),extent,km2,6046972.7,6046972.7,0
Circum-Arctic (NH),volume,km3,12410.703,12410.703,0
Circum-Arctic (NH),mean_ice_thickness,m,2.681286,2.681286,0
Circum-Antarctic (SH),area,km2,17059539,17059539,0
Circum-Antarctic (SH),extent,km2,19618958,19618958,0
Circum-Antarctic (SH),volume,km3,15894.744,15894.744,0
Circum-Antarctic (SH),mean_ice_thickness,m,0.93172174,0.93172174,0


CSV: /g/data/gv90/da1339/src/CICE_dyntens/validation_report/global/g0_summary/g0-regional-totals.csv


## Available strength and deformation fields

Means and standard deviations use ice-area weights (cell area × daily mean concentration), selecting cells with concentration at least the threshold. Minimum and maximum are over valid selected cells. The valid-ice-area fractions show whether masking reduced coverage.

The largest cell difference and unequal-cell counts supplement the regional statistics: equal means alone could hide differences that cancel. `hs` and `Tsfc` were not written in these runs; missing fields are listed above. Fields on velocity grids are not mixed with T-cell weights or silently interpolated. Units come from the actual history variables.


In [5]:
fields = show_csv(output / "g0-regional-fields.csv")
print("Summary record:", output / "g0-regional-summary.json")


region,variable,statistic,units,off,unity,unity_minus_off,max_abs_cell_difference,unequal_cells,masks_equal,off_valid_ice_area_fraction,unity_valid_ice_area_fraction
Circum-Arctic (NH),strength,minimum,N/m,0.00011261977,0.00011261977,0,0,0,True,1,1
Circum-Arctic (NH),strength,mean,N/m,9922.3085,9922.3085,0,0,0,True,1,1
Circum-Arctic (NH),strength,maximum,N/m,196724.74,196724.74,0,0,0,True,1,1
Circum-Arctic (NH),strength,std,N/m,11421.489,11421.489,0,0,0,True,1,1
Circum-Arctic (NH),divu,minimum,%/day,-393.86493,-393.86493,0,0,0,True,1,1
Circum-Arctic (NH),divu,mean,%/day,-0.34310806,-0.34310806,0,0,0,True,1,1
Circum-Arctic (NH),divu,maximum,%/day,521.18964,521.18964,0,0,0,True,1,1
Circum-Arctic (NH),divu,std,%/day,10.223699,10.223699,0,0,0,True,1,1
Circum-Arctic (NH),shear,minimum,%/day,0.0020596405,0.0020596405,0,0,0,True,1,1
Circum-Arctic (NH),shear,mean,%/day,6.11222,6.11222,0,0,0,True,1,1


CSV: /g/data/gv90/da1339/src/CICE_dyntens/validation_report/global/g0_summary/g0-regional-fields.csv
Summary record: /g/data/gv90/da1339/src/CICE_dyntens/validation_report/global/g0_summary/g0-regional-summary.json


## Interpretation

For the completed G0 pair, every reported difference should be zero. The full exact-file comparison remains the evidence of equality; these tables present selected regional results.

The comparison shows that switching on the constant unity coefficient left this two-day global solution unchanged. The thermal restart variables also matched. It does not establish seasonal performance, an independent heat budget, WHACS loading or the behaviour of a changing FSD-based coefficient. Those questions remain for G1/G2.
